In [1]:
import pandas as pd
import numpy as np
import sys, os
from helpers import Condition, Participant, Trial
from helpers.metadata import AVOIDING_THRESHOLD,PIXEL_RAILWAY_WIDTH, PIXEL_RAIL_RIGHT, PIXEL_RAIL_LEFT

In [2]:
ai = Condition.load_conditional_group(condition="ai")
ad = Condition.load_conditional_group(condition="ad")

In [3]:
def transform_to_normalized_distances(trial: Trial) -> Trial:
    """Nomalize to start = 0 and target = 1"""

    tx, ty = trial.get_target_in_cm()
    sx, _ = trial.get_start()
    trajectory_px = trial.get_trajectory_data()[["current_pos_x"]]

    trial[["current_pos_x", "current_pos_y"]] = trial.get_trajectory_data_in_cm()[["current_pos_x", "current_pos_y"]]
    trial[["current_pos_x_normalized", "current_pos_y_normalized"]] = trial.get_trajectory_data_normalized()[["current_pos_x", "current_pos_y"]]
    trial[["target_pos_x"]] = tx
    trial[["target_pos_y"]] = ty
    trial["trial_rail_pos"] = (trajectory_px + np.abs(sx)) / PIXEL_RAILWAY_WIDTH

    return trial

In [4]:
threshold = AVOIDING_THRESHOLD * PIXEL_RAILWAY_WIDTH
def add_trajectory_side(trial:
     Trial) -> Trial:
    """appends trial dataframe by column to store whether side was switched"""

    trial["switched_side"] = 0 
    trial.loc[trial["trial_rail_pos"] >= AVOIDING_THRESHOLD, "switched_side"] = 1
    return trial

In [5]:
def drop_unneccessary(trial: Trial):
    trial = trial.drop(["left_button_pressed","middle_button_pressed","right_button_pressed"], axis=1)
    return trial

In [6]:
#def normalize_time(data):
#    """
#    Normalize the time frame to go from 0 to 1.
#    Interpolate the data to get 10ms time steps
#    """

#    data.loc[:,'datetime'] = pd.date_range('1/1/2001 00:00:00', '1/1/2001 00:00:01',len(data))
#    normdata = data.set_index('datetime', drop = True).resample('10ms').mean().interpolate()
#    normdata['normtime'] = np.arange(0,1.01,0.01)
#    normdata = normdata.reset_index(drop=True)

#    return normdata

In [7]:
def process(trial:Trial) -> Trial:
    trial = transform_to_normalized_distances(trial)
    trial = add_trajectory_side(trial)
    trial = drop_unneccessary(trial)
    return trial


In [8]:
FILE_PATH = "preprocessed_avoiding"
if not os.path.exists(FILE_PATH):
    os.mkdir(FILE_PATH)

for condition in [ai, ad]:

    p_processed = []
    for participant in condition:
        sys.stdout.write(f"\rProcessing {participant.get_participant_id()}\t")
        p_processed += list(map(process, participant))

    sys.stdout.write(f"\rConcatinating {condition.get_group_name()}\t")
    df_joint = pd.concat(p_processed).round(3)
    df_joint.to_csv(f"{FILE_PATH}/{condition.get_group_name()}.csv")


sys.stdout.write(f"\nDone")
sys.stdout.flush()

Concatinating ad	
Done